# <font color="#418FDE" size="6.5" uppercase>**Thresholds Curves**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Tune classification thresholds using probabilities or decision scores. 
- Select thresholds based on practical objectives while avoiding leakage. 
- Use validation and learning curves to diagnose bias, variance, and scalability. 


## **1. Threshold Basics**

### **1.1. Probability Scores**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_01_01.jpg?v=1784036005" width="250">



>* Scores show strength of class evidence
>* Thresholds turn scores into class labels

>* Scores may not be calibrated probabilities
>* Calibration affects threshold confidence decisions

>* Adjust thresholds to match error costs
>* Choose cutoffs aligned with real goals



In [ ]:
#@title Python Code - Probability Scores

# This example tunes labels using probability scores.
# Different thresholds create different practical tradeoffs.
# The plot shows precision and recall changing.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

# Load a small binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Check that the target has exactly two classes.
class_count = len(np.unique(y))
if class_count != 2:
    raise ValueError("This example needs a binary target.")

# Split before training to avoid evaluation leakage.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Scale features inside the pipeline using training data only.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

model.fit(X_train, y_train)
positive_scores = model.predict_proba(X_valid)[:, 1]

# Convert probability scores into labels at several thresholds.
thresholds = np.linspace(0.05, 0.95, 19)
precisions = []
recalls = []

for threshold in thresholds:
    predicted_positive = positive_scores >= threshold
    precision = precision_score(y_valid, predicted_positive, zero_division=0)
    recall = recall_score(y_valid, predicted_positive, zero_division=0)
    precisions.append(precision)
    recalls.append(recall)

# Print a few concise results before plotting the full curve.
print("scikit-learn version:", sklearn.__version__)
print("Threshold 0.30 recall:", round(recalls[5], 3))
print("Threshold 0.70 precision:", round(precisions[13], 3))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, precisions, marker="o", label="Precision")
ax.plot(thresholds, recalls, marker="o", label="Recall")

ax.set_title("Probability Thresholds Change Classification Behavior")
ax.set_xlabel("Probability threshold for the positive class")
ax.set_ylabel("Validation score")
ax.set_ylim(0, 1.05)

ax.legend()
plt.show()



### **1.2. Default Decision Thresholds**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_01_02.jpg?v=1784036007" width="250">



>* Thresholds convert scores into class labels
>* Default cutoffs assume equal costs and calibration

>* Default thresholds simplify early model comparison
>* Real-world costs require careful threshold choices

>* Scores differ across models and calibration quality
>* Tune thresholds to match real-world consequences



In [ ]:
#@title Python Code - Default Decision Thresholds

# This example shows default classification thresholds.
# Probabilities become labels at a chosen cutoff.
# Changing the cutoff changes practical model behavior.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Confirm the dataset has matching rows and labels.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and labels must match.")

# Split before fitting to avoid test-set leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Fit scaling and logistic regression only on training data.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Predict probabilities for the positive class on held-out data.
positive_probabilities = model.predict_proba(X_test)[:, 1]
thresholds = np.array([0.30, 0.50, 0.70])

# Compare how three thresholds change the confusion matrix.
print("scikit-learn version:", sklearn.__version__)
print("Threshold | TP | FP | FN | TN")
for threshold in thresholds:
    predicted_labels = positive_probabilities >= threshold
    tn, fp, fn, tp = confusion_matrix(y_test, predicted_labels).ravel()
    print(f"{threshold:.2f}      | {tp:2d} | {fp:2d} | {fn:2d} | {tn:2d}")

# Plot probabilities with the default threshold as a hard boundary.
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(positive_probabilities[y_test == 0], bins=12, alpha=0.7, label="Class 0")
ax.hist(positive_probabilities[y_test == 1], bins=12, alpha=0.7, label="Class 1")

ax.axvline(0.50, color="black", linestyle="--", label="Default threshold")
ax.set_title("Default threshold turns probabilities into labels")
ax.set_xlabel("Predicted probability for class 1")
ax.set_ylabel("Number of test examples")
ax.legend()
plt.show()



### **1.3. Cross Validation Thresholds**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_01_03.jpg?v=1784036009" width="250">



>* Cross validation reduces split-based threshold luck
>* Multiple folds reveal more reliable cutoffs

>* Tune thresholds to match performance goals
>* Cross validation checks tradeoffs across folds

>* Separate threshold tuning from final testing
>* Use validation to estimate real-world performance



In [ ]:
#@title Python Code - Cross Validation Thresholds

# Cross validation chooses a stable threshold.
# Validation folds test candidate decision cutoffs.
# The final test set stays untouched.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Check that the example is binary classification.
if len(np.unique(y)) != 2:
    raise ValueError("This lesson needs exactly two classes.")

# Keep the test set separate until the end.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Candidate thresholds turn probabilities into class labels.
thresholds = np.linspace(0.10, 0.90, 17)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
mean_f1_scores = []

# Evaluate each threshold only on validation folds.
for threshold in thresholds:
    fold_scores = []
    for train_index, valid_index in cv.split(X_train, y_train):
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, random_state=42)
        )
        model.fit(X_train[train_index], y_train[train_index])
        valid_prob = model.predict_proba(X_train[valid_index])[:, 1]
        valid_pred = (valid_prob >= threshold).astype(int)
        fold_scores.append(f1_score(y_train[valid_index], valid_pred))
    mean_f1_scores.append(float(np.mean(fold_scores)))

# Select the threshold with the best cross validated score.
best_index = int(np.argmax(mean_f1_scores))
best_threshold = float(thresholds[best_index])
best_cv_f1 = float(mean_f1_scores[best_index])

# Fit once on all training data, then test once.
final_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)
final_model.fit(X_train, y_train)
test_prob = final_model.predict_proba(X_test)[:, 1]

default_test_f1 = f1_score(y_test, (test_prob >= 0.50).astype(int))
tuned_test_f1 = f1_score(y_test, (test_prob >= best_threshold).astype(int))

print("scikit-learn version:", sklearn.__version__)
print("Best CV threshold:", round(best_threshold, 2))
print("Mean CV F1 at best threshold:", round(best_cv_f1, 3))
print("Test F1 at 0.50 threshold:", round(default_test_f1, 3))
print("Test F1 at tuned threshold:", round(tuned_test_f1, 3))

# Plot how validation performance changes with the threshold.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, mean_f1_scores, marker="o", label="Mean CV F1")
ax.axvline(best_threshold, color="red", linestyle="--", label="Chosen threshold")
ax.set_title("Cross Validated Threshold Selection")
ax.set_xlabel("Classification threshold")
ax.set_ylabel("Mean validation F1 score")
ax.legend()
plt.show()



## **2. Threshold Design**

### **2.1. Fixed Thresholds**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_02_01.jpg?v=1784035999" width="250">



>* Thresholds convert scores into decisions
>* Default cutoffs may not fit every problem

>* Fixed thresholds provide consistent, simple decisions
>* Choose thresholds based on error consequences

>* Choose thresholds before final testing
>* Keep test data untouched to avoid leakage



In [ ]:
#@title Python Code - Fixed Thresholds

# Fixed thresholds turn scores into decisions.
# Validation data chooses the threshold safely.
# Test data estimates final future performance.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

# Create a small imbalanced classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=8,
    n_informative=5,
    weights=[0.85, 0.15],
    random_state=42,
)

# Split once for final testing, then split training for validation.
features_train_full, features_test, target_train_full, target_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

features_train, features_valid, target_train, target_valid = train_test_split(
    features_train_full, target_train_full, test_size=0.30, stratify=target_train_full,
    random_state=42
)

# Fit the model without looking at validation or test labels.
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(features_train, target_train)

# Use validation probabilities to choose one fixed threshold.
valid_scores = model.predict_proba(features_valid)[:, 1]
thresholds = np.linspace(0.05, 0.95, 19)
valid_f1_scores = []

for threshold in thresholds:
    valid_predictions = valid_scores >= threshold
    score = f1_score(target_valid, valid_predictions)
    valid_f1_scores.append(score)

best_index = int(np.argmax(valid_f1_scores))
best_threshold = float(thresholds[best_index])

# Evaluate the chosen threshold once on untouched test data.
test_scores = model.predict_proba(features_test)[:, 1]
test_predictions = test_scores >= best_threshold

precision = precision_score(target_test, test_predictions)
recall = recall_score(target_test, test_predictions)
test_f1 = f1_score(target_test, test_predictions)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Chosen validation threshold: {best_threshold:.2f}")
print(f"Untouched test precision: {precision:.2f}")
print(f"Untouched test recall: {recall:.2f}")
print(f"Untouched test F1 score: {test_f1:.2f}")

# Plot validation F1 to show threshold selection before testing.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, valid_f1_scores, marker="o", label="Validation F1")
ax.axvline(best_threshold, color="red", linestyle="--", label="Chosen threshold")

ax.set_title("Choosing a Fixed Threshold on Validation Data")
ax.set_xlabel("Fixed probability threshold")
ax.set_ylabel("Validation F1 score")
ax.legend()
plt.show()



### **2.2. Objective Driven Thresholds**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_02_02.jpg?v=1784036001" width="250">



>* Thresholds turn model scores into actions
>* Choose cutoffs based on real-world costs

>* Optimize sensitivity, precision, or workload goals
>* Choose thresholds for real decision needs

>* Consider changing costs across contexts
>* Validate thresholds with experts and real data



In [ ]:
#@title Python Code - Objective Driven Thresholds

# This example chooses thresholds for practical objectives.
# Validation data prevents threshold selection leakage.
# The plot shows objective driven tradeoffs.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

from sklearn.preprocessing import StandardScaler

# Load a small packaged binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Treat malignant cases as the positive class.
y = 1 - y

# Check the dataset has matching rows and labels.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and labels must match.")

# Split before modeling to keep validation decisions honest.
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=42
)

# Fit preprocessing only on training data through a pipeline.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Validation probabilities are scores, not final decisions.
valid_scores = model.predict_proba(X_valid)[:, 1]
thresholds = np.linspace(0.05, 0.95, 91)

# Evaluate each possible policy threshold on validation data.
recalls = []
precisions = []
for threshold in thresholds:
    predictions = valid_scores >= threshold
    recalls.append(recall_score(y_valid, predictions, zero_division=0))
    precisions.append(precision_score(y_valid, predictions, zero_division=0))

# Choose the highest threshold that still meets recall needs.
recalls = np.array(recalls)
precisions = np.array(precisions)
valid_choices = thresholds[recalls >= 0.95]

# Validate that the objective is achievable on validation data.
if len(valid_choices) == 0:
    raise ValueError("No threshold reached the required validation recall.")

chosen_threshold = valid_choices[-1]
chosen_index = np.where(thresholds == chosen_threshold)[0][0]
chosen_recall = recalls[chosen_index]
chosen_precision = precisions[chosen_index]

print("scikit-learn version:", sklearn.__version__)
print("Objective: recall at least 0.95 on validation data")
print("Chosen threshold:", round(float(chosen_threshold), 2))
print("Validation recall:", round(float(chosen_recall), 3))
print("Validation precision:", round(float(chosen_precision), 3))

# Plot how the objective changes as the threshold moves.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, recalls, label="Recall")
ax.plot(thresholds, precisions, label="Precision")

ax.axvline(chosen_threshold, color="black", linestyle="--", label="Chosen")
ax.set_title("Validation Metrics Across Thresholds")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Validation score")
ax.legend()

plt.show()



### **2.3. Avoid Threshold Leakage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_02_03.jpg?v=1784036003" width="250">



>* Test data must not choose thresholds
>* Treat thresholds as model selection

>* Choose thresholds using validation data
>* Use test data once for honest evaluation

>* Use only information available at decision time
>* Validate thresholds on genuinely unseen data



In [ ]:
#@title Python Code - Avoid Threshold Leakage

# This example shows threshold leakage in classification.
# Validation data chooses the decision threshold safely.
# Test data estimates final performance only once.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small binary classification dataset.
features, labels = make_classification(
    n_samples=1200,
    n_features=8,
    n_informative=4,
    n_redundant=2,
    weights=[0.7, 0.3],
    flip_y=0.08,
    random_state=42,
)

# Split once for training and once for honest testing.
train_features, temp_features, train_labels, temp_labels = train_test_split(
    features,
    labels,
    test_size=0.4,
    stratify=labels,
    random_state=42,
)

# Split temporary data into validation and test sets.
valid_features, test_features, valid_labels, test_labels = train_test_split(
    temp_features,
    temp_labels,
    test_size=0.5,
    stratify=temp_labels,
    random_state=42,
)

# Fit preprocessing only on the training data.
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features)
valid_scaled = scaler.transform(valid_features)
test_scaled = scaler.transform(test_features)

# Train one simple probability model.
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(train_scaled, train_labels)

# Get probabilities for threshold selection and evaluation.
valid_probabilities = model.predict_proba(valid_scaled)[:, 1]
test_probabilities = model.predict_proba(test_scaled)[:, 1]

# Check that probabilities match their labels.
if len(valid_probabilities) != len(valid_labels):
    raise ValueError("Validation probabilities and labels must match.")

# Try a small, readable set of candidate thresholds.
thresholds = np.linspace(0.1, 0.9, 17)
valid_scores = []
test_scores = []

# Score each threshold on validation and test data.
for threshold in thresholds:
    valid_predictions = valid_probabilities >= threshold
    test_predictions = test_probabilities >= threshold
    valid_scores.append(f1_score(valid_labels, valid_predictions))
    test_scores.append(f1_score(test_labels, test_predictions))

# Choose the threshold using validation data only.
best_valid_index = int(np.argmax(valid_scores))
safe_threshold = float(thresholds[best_valid_index])
safe_test_score = float(test_scores[best_valid_index])

# This leaky choice incorrectly uses test labels.
best_test_index = int(np.argmax(test_scores))
leaky_threshold = float(thresholds[best_test_index])
leaky_test_score = float(test_scores[best_test_index])

print("scikit-learn version:", sklearn.__version__)
print("Safe validation-chosen threshold:", round(safe_threshold, 2))
print("Honest test F1 at safe threshold:", round(safe_test_score, 3))
print("Leaky test-chosen threshold:", round(leaky_threshold, 2))
print("Over-optimistic test F1 after leakage:", round(leaky_test_score, 3))



## **3. Diagnostic Curves**

### **3.1. Validation Curves**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_03_01.jpg?v=1784035993" width="250">



>* Compare training and validation across tuning values
>* Spot bias, variance, and balanced complexity

>* Track complexity to spot underfitting
>* Detect overfitting before trusting models

>* Tune with separate validation data, not tests
>* Compare curves to diagnose reliability



In [ ]:
#@title Python Code - Validation Curves

# This script demonstrates a validation curve.
# Tree depth controls model flexibility here.
# The plot reveals underfitting and overfitting.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import validation_curve
from sklearn.tree import DecisionTreeClassifier

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and target labels must match.")

# Try several tree depths from simple to flexible.
depths = np.arange(1, 11)
model = DecisionTreeClassifier(random_state=42)

# Cross-validation keeps validation scores separate from fitting.
train_scores, validation_scores = validation_curve(
    model, X, y, param_name="max_depth", param_range=depths, cv=5
)

# Average the fold scores for a clear beginner-friendly curve.
train_mean = train_scores.mean(axis=1)
validation_mean = validation_scores.mean(axis=1)

# Identify the depth with the best validation performance.
best_index = int(np.argmax(validation_mean))
best_depth = int(depths[best_index])
best_score = float(validation_mean[best_index])

print("scikit-learn version:", sklearn.__version__)
print("Best validation depth:", best_depth)
print("Best validation accuracy:", round(best_score, 3))

# Plot training and validation accuracy against model complexity.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths, train_mean, marker="o", label="Training accuracy")
ax.plot(depths, validation_mean, marker="o", label="Validation accuracy")

ax.set_title("Validation Curve for Decision Tree Depth")
ax.set_xlabel("Maximum tree depth")
ax.set_ylabel("Accuracy")
ax.set_xticks(depths)
ax.legend()

plt.show()



### **3.2. Learning Curves**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_03_02.jpg?v=1784035997" width="250">



>* Compare training and validation performance as data grows
>* Spot underfitting, overfitting, and improvement limits

>* Improving validation curves justify more data
>* Flat curves suggest pipeline or model changes

>* Prevent leakage with separate validation data
>* Match validation design to real deployment



In [ ]:
#@title Python Code - Learning Curves

# Learning curves compare training and validation performance.
# Curve gaps diagnose bias, variance, and data value.
# This example plots scores as training size grows.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import learning_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows and target labels must match.")

# A pipeline prevents scaling leakage during cross-validation.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

# Stratified folds keep class proportions similar in each split.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sizes = np.linspace(0.1, 1.0, 6)

# Learning curves refit the model on increasing training sizes.
sizes, train_scores, validation_scores = learning_curve(
    model, X, y, train_sizes=train_sizes, cv=cv, scoring="accuracy"
)

# Average the fold scores for a beginner-friendly plot.
train_mean = train_scores.mean(axis=1)
validation_mean = validation_scores.mean(axis=1)

# Print only concise context for the plotted diagnostic.
print("scikit-learn version:", sklearn.__version__)
print("Largest training score:", round(float(train_mean[-1]), 3))
print("Largest validation score:", round(float(validation_mean[-1]), 3))

# Plot one learning curve figure for diagnosis.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sizes, train_mean, marker="o", label="Training accuracy")
ax.plot(sizes, validation_mean, marker="o", label="Validation accuracy")

# Labels make the diagnostic meaning explicit.
ax.set_title("Learning Curve for Logistic Regression")
ax.set_xlabel("Number of training examples")
ax.set_ylabel("Accuracy")
ax.legend()

# A grid helps compare the gap between curves.
ax.grid(True, alpha=0.3)
plt.show()



### **3.3. Scalability Diagnosis**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_28/Lecture_B/image_03_03.jpg?v=1784035995" width="250">



>* Measure performance as data grows
>* Balance accuracy with deployment costs

>* Learning curves show if more data helps
>* Balance performance gains against added costs

>* Thresholds affect workload at larger scale
>* Use validation curves for deployment planning



In [ ]:
#@title Python Code - Scalability Diagnosis

# Diagnose scalability with a small learning curve.
# Compare validation score against training cost.
# See when more data gives diminishing returns.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a deterministic binary classification dataset.
features, target = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    random_state=42,
)

# Split once so validation data stays untouched.
train_features, valid_features, train_target, valid_target = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Validate the basic shape before modeling.
if train_features.shape[0] < 1000:
    raise ValueError("Need at least 1000 training rows.")

# Fit preprocessing only on the training split.
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_features)
valid_scaled = scaler.transform(valid_features)

# Train the same model on increasing data sizes.
train_sizes = np.array([200, 500, 1000, 2000, 3500])
validation_auc = []
training_cost_units = []

for size in train_sizes:
    model = LogisticRegression(max_iter=300, random_state=42)
    model.fit(train_scaled[:size], train_target[:size])
    valid_scores = model.predict_proba(valid_scaled)[:, 1]
    validation_auc.append(roc_auc_score(valid_target, valid_scores))
    training_cost_units.append(size * train_scaled.shape[1])

# Print concise results for beginners.
print("scikit-learn version:", sklearn.__version__)
print("Rows used:", train_sizes.tolist())
print("Validation AUC:", [round(score, 3) for score in validation_auc])

# Plot performance against a simple cost proxy.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(training_cost_units, validation_auc, marker="o", label="Validation AUC")
ax.set_title("Scalability Diagnosis: More Data vs Benefit")
ax.set_xlabel("Training cost proxy: rows x features")
ax.set_ylabel("Validation ROC AUC")
ax.legend()

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Thresholds Curves**</font>


In this lecture, you learned to:
- Tune classification thresholds using probabilities or decision scores. 
- Select thresholds based on practical objectives while avoiding leakage. 
- Use validation and learning curves to diagnose bias, variance, and scalability. 

In the next Module (Module 29), we will go over 'Metrics Scoring'